In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_style('whitegrid', rc={'ytick.left': True, 'ytick.color': 'silver'})
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['axes.autolimit_mode'] = 'round_numbers'
plt.rcParams['ytick.labelcolor'] = 'black'
sns.set_palette('colorblind')

In [ ]:
df = pd.read_csv('gov.csv')
# KEY = 'RAPL package-0 (J)'
KEY = 'INA system energy (J)'
df['sort'] = df.groupby('url')[KEY].transform('mean')
df = df.sort_values(by='sort', ascending=False).drop('sort', axis=1).reset_index(drop=True)

## Data distribution

In [ ]:
df_mean = df.groupby('url').mean()
df_mean.describe()

In [ ]:
ax = sns.histplot(df_mean, x=KEY, bins=20, color='#3DA542', alpha=1)

c = ax.containers[0]
bins = [rect.get_height() for rect in c]
ax.bar_label(c, color='k', fontsize=8)

plt.xlim(0,None)
plt.axvline(x=df_mean[KEY].mean(), ymax=.9, color='#A9A9A9')
plt.axvline(x=df_mean[KEY].median(), ymax=.9, color='#A9A9A9')
plt.xlabel('Energieverbruik (Joule)')
plt.tight_layout()
plt.show()

## Top-10

In [ ]:
def squash(s):
    ''' Replace everthing between outermost `/` with dots. '''
    import re
    return re.sub(r'(?<=/).*(?=/)', '...', s) if s.count('/') >= 2 else s

In [ ]:
N_TOP, N_BOT = 10, 5
ASPECT = N_TOP / N_BOT
top, bot = df[:(5*N_TOP)], df[-(5*N_BOT):]

fig, (ax0, ax1) = plt.subplots(2, sharex=True, height_ratios=[N_TOP,N_BOT], figsize=(10,5.5))
sns.barplot(y=list(map(squash, top['url'])), x=top[KEY], orient='h', color='#3DA542', ax=ax0)
sns.barplot(y=list(map(squash, bot['url'])), x=bot[KEY], orient='h', color='#3DA542', ax=ax1)
ax1.set_xlabel('Energieverbruik (Joule)')
plt.xlim(0,60)

# Hide spines between axes
ax0.spines['bottom'].set_visible(False)
ax1.spines['top'].set_visible(False)

# Add diagonal break marks
d = .015
kwargs = dict(transform=ax0.transAxes, color='silver', clip_on=False)
ax0.plot((0-d, 0+d), (0-d, 0+d), **kwargs) # left
ax0.plot((1-d, 1+d), (0-d, 0+d), **kwargs) # right
kwargs.update(transform=ax1.transAxes)
ax1.plot((0-d, 0+d), (1-d*ASPECT, 1+d*ASPECT), **kwargs) # left
ax1.plot((1-d, 1+d), (1-d*ASPECT, 1+d*ASPECT), **kwargs) # right

plt.tight_layout()
plt.savefig('rank.svg', bbox_inches='tight', transparent=True)
plt.show()

## Savings

Assume that all websites above the average are optimised, so that their energy consumption matches the average.

In [ ]:
# Rough estimates
VISITS_PER_YEAR = 50_000
TIME_PER_VISIT = 5 * 60
CHARGE_PER_PHONE = 50_000

In [ ]:
ub = df_mean[KEY].mean()
print(len(df_mean[df_mean[KEY] > ub]), 'above-average sites')
# Divide by 4 because we measure for 4 seconds
savings = (df_mean[KEY].mean() - df_mean[KEY].clip(upper=ub).mean()) / 4
print(f'Savings/visit = {savings:.3}W')

total_visit_time = len(df_mean) * VISITS_PER_YEAR * TIME_PER_VISIT
savings_per_year = savings * total_visit_time
charges_per_year = savings_per_year / CHARGE_PER_PHONE
print(f'Charges/year = {int(charges_per_year.round())}')